# Fairness Analysis (Phase 5)

Compare fairness metrics between standard KG2RAG and fairness-aware KG2RAG.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
ROOT = Path.cwd().parent
metrics_dir = ROOT / 'outputs' / 'metrics'
split = 'dev'

experiments = ['kg2rag_standard', 'kg2rag_fair']
fair_rows = []

for exp in experiments:
    path = metrics_dir / f'{exp}_metrics_{split}.json'
    if not path.exists():
        print(f'Missing metrics for {exp}: {path}')
        continue

    payload = pd.read_json(path, typ='series')
    fairness = payload.get('fairness', {})

    for attr_name, attr_vals in fairness.items():
        if not isinstance(attr_vals, dict):
            continue
        fair_rows.append({
            'experiment': exp,
            'attribute': attr_name,
            'parity_diff': attr_vals.get('demographic_parity_diff'),
            'impact_ratio': attr_vals.get('disparate_impact_ratio'),
            'equalized_odds_diff': attr_vals.get('equalized_odds_diff'),
        })

fair_df = pd.DataFrame(fair_rows)
fair_df

In [ ]:
if not fair_df.empty:
    plot_df = fair_df.melt(id_vars=['experiment', 'attribute'], value_vars=['parity_diff', 'equalized_odds_diff', 'impact_ratio'], var_name='metric', value_name='value')

    g = sns.catplot(
        data=plot_df,
        x='metric', y='value', hue='experiment', col='attribute',
        kind='bar', height=4, aspect=1.0
    )
    g.fig.subplots_adjust(top=0.85)
    g.fig.suptitle('Fairness Metric Comparison')
    plt.show()
else:
    print('No fairness metrics found yet. Run experiments first.')

In [ ]:
per_q_path = metrics_dir / f'kg2rag_fair_per_question_{split}.json'
if per_q_path.exists():
    per_q = pd.read_json(per_q_path)
    if {'geo_group', 'exact_match'}.issubset(per_q.columns):
        summary = per_q.groupby('geo_group', dropna=False)['exact_match'].mean().sort_values(ascending=False)
        display(summary.to_frame('exact_match_rate'))

        plt.figure(figsize=(8, 4))
        sns.barplot(x=summary.index.astype(str), y=summary.values)
        plt.title('Fair KG2RAG: Exact-Match by Geo Group')
        plt.ylabel('Exact-Match Rate')
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()
    else:
        print('Per-question file exists but required columns are missing.')
else:
    print(f'Missing {per_q_path}. Run scripts/run_experiment.py --experiment kg2rag_fair first.')